Script de extração dos dados via API

In [3]:
# Import das bibliotecas
import requests
import boto3
import os
import logging
import json
from botocore.client import Config
from botocore.exceptions import ClientError
from conf.settings import settings
from datetime import datetime


# Paths
GITHUB_URL = settings.github_url
MINIO_ENDPOINT = settings.minio_endpoint
BUCKET_NAME = settings.minio_bucket
BRONZE_DATA = settings.bronze_data
BRONZE_METADATA = settings.bronze_metadata
MINIO_ACCESS_KEY = settings.minio_access_key
MINIO_SECRET_KEY = settings.minio_secret_key

# Configuração de Log
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Função de conexão do MiniO via boto3
def conexao_minio_client():
    return boto3.client(
        "s3",
        endpoint_url=MINIO_ENDPOINT,
        aws_access_key_id=MINIO_ACCESS_KEY,
        aws_secret_access_key=MINIO_SECRET_KEY,
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )

# Função para verificar se bucket existe, senão cria
def garantir_bucket(s3):
    try:
        s3.head_bucket(Bucket=BUCKET_NAME)
    except ClientError:
        print(f"Criando bucket: {BUCKET_NAME}")
        s3.create_bucket(Bucket=BUCKET_NAME)

# Função para extração dos dados via API
def extrair_dados():
    logging.info("Iniciando extração...")
    
    # Partição por data e id de execução
    ingestion_date = datetime.utcnow().strftime("%Y-%m-%d")
    execution_id = datetime.utcnow().strftime("%Y%m%dT%H%M%S")

    arquivos_enviados = []
    status = "success"
    error_message = None
    
    try:
    
        response = requests.get(GITHUB_URL, timeout=10)
    
        if response.status_code != 200:
            raise Exception("Erro ao acessar a API.")

        arquivos = response.json()
        s3 = conexao_minio_client()
        garantir_bucket(s3)

        # Loop para ingestão dos arquivos
        for arquivo in arquivos:
            nome = arquivo["name"]
            download_url = arquivo["download_url"]

            chave = (
                f"{BRONZE_DATA}raw/"
                f"ingestion_date={ingestion_date}/"
                f"execution_id={execution_id}/"
                f"{nome}"
            )

            logging.info(f"Processando {nome}...")

            file_response = requests.get(download_url, timeout=10)

            # Verificar se o retorno é OK
            if file_response.status_code == 200:
                s3.put_object(
                    Bucket=BUCKET_NAME,
                    Key=chave,
                    Body=file_response.content,
                )
                arquivos_enviados.append(nome)
            else:
                logging.warning(f"Erro ao baixar {nome}")

        logging.info(f"{len(arquivos_enviados)} arquivos enviados")

    except Exception as e:
        status = "failed"
        error_message = str(e)
        
        logging.error(f"Falha na execução: {error_message}")
                      
    finally:
    
        # Metadados por execução na camada Bronze (Governaça de dados)
        metadata = {
            "execution_id": execution_id,
            "pipeline": "github_ingestion",
            "source": "github_api",
            "ingestion_date": ingestion_date,
            "execution_timestamp": datetime.utcnow().isoformat(),
            "total_files": len(arquivos_enviados),
            "status": status,
            "error_message": error_message
        }
    
        metadata_key = f"{BRONZE_METADATA}execution_id={execution_id}/metadata.json"
                      
        s3.put_object(
            Bucket=BUCKET_NAME,
            Key=metadata_key,
            Body=json.dumps(metadata, indent=4),
            ContentType="application/json"
        )
                      
        logging.info("Metadata da execução salvo")
    
    return arquivos_enviados
        

In [4]:
# Extrair os dados para o Data lake
extrair_dados()

2026-03-06 00:50:23,393 - INFO - Iniciando extração...
2026-03-06 00:50:23,798 - INFO - Processando Aaron697_Brekke496_2fa15bc7-8866-461a-9000-f739e425860a.json...
2026-03-06 00:50:23,977 - INFO - Processando Aaron697_Stiedemann542_41166989-975d-4d17-b9de-17f94cb3eec1.json...
2026-03-06 00:50:24,162 - INFO - Processando Abby752_Kuvalis369_2b083021-e93f-4991-bf49-fd4f20060ef8.json...
2026-03-06 00:50:24,330 - INFO - Processando Abel832_Connelly992_29e51479-f742-4474-8f8e-d2607d5269f6.json...
2026-03-06 00:50:36,055 - INFO - Processando Abraham100_Heller342_262b819a-5193-404a-9787-b7f599358035.json...
2026-03-06 00:50:24,695 - INFO - Processando Adam631_Cronin387_aff8f143-2375-416f-901d-b0e4c73e3e58.json...
2026-03-06 00:50:24,877 - INFO - Processando Adam631_Shields502_9e2653fc-49e0-4b2e-86f8-e664bbe07be3.json...
2026-03-06 00:50:25,065 - INFO - Processando Adelaida985_Gulgowski816_86662c9c-fcc8-41a6-a2e8-61270bf8a3b0.json...
2026-03-06 00:50:25,254 - INFO - Processando Adolfo777_O'Keef

2026-03-06 00:50:37,650 - INFO - Processando April374_Gerhold939_e3c6cb9c-5f13-4d78-bc38-d7c005346389.json...
2026-03-06 00:50:37,823 - INFO - Processando Archie818_Barrows492_fef3d64f-82b0-47e0-be02-3317c4969826.json...
2026-03-06 00:50:37,998 - INFO - Processando Ardath226_Zboncak558_cb96223c-c32b-45be-8aa7-36795883a060.json...
2026-03-06 00:50:38,176 - INFO - Processando Ardelia466_Larson43_68b95ae3-8732-4948-9345-f3d490bc4ae5.json...
2026-03-06 00:50:38,349 - INFO - Processando Arica110_Herman763_c08a8771-4cfb-458f-a248-d4dad06a6e17.json...
2026-03-06 00:50:38,533 - INFO - Processando Ariel183_Friesen796_1ae87f34-f0c9-4312-91cd-349dd9b790b7.json...
2026-03-06 00:50:38,701 - INFO - Processando Ariel183_Turner526_6edf069d-ebad-4c90-8192-119fb32c84c1.json...
2026-03-06 00:50:38,878 - INFO - Processando Arielle168_Marvin195_c76c71f4-7193-4d78-9e36-c99011243b72.json...
2026-03-06 00:50:39,068 - INFO - Processando Armand155_Mills423_9e36e8d9-b864-453f-a2ef-6eac2df7f080.json...
2026-03-06

2026-03-06 00:50:52,004 - INFO - Processando Bud153_Lang846_83cd3265-e2b8-4a84-95fb-491738d4fd75.json...
2026-03-06 00:50:52,192 - INFO - Processando Buffy238_Schumm995_ccb86a73-115d-4f69-8c13-50083e549438.json...
2026-03-06 00:50:52,382 - INFO - Processando Buffy238_Wiza601_a99b82af-6654-4101-8091-cea62bae7826.json...
2026-03-06 00:50:52,570 - INFO - Processando Bunny174_Sauer652_3038f107-f43e-4f8d-bee3-0dd51236ac73.json...
2026-03-06 00:50:52,755 - INFO - Processando Burl285_Torp761_2b4109b0-0a81-409d-ac15-ecc25bb91175.json...
2026-03-06 00:50:52,923 - INFO - Processando Burma963_Oberbrunner298_f42c7e4c-fc13-4aa7-9d58-352d31ba9105.json...
2026-03-06 00:50:53,144 - INFO - Processando Buster609_Durgan499_d8c908da-de6b-41da-8abf-96b27df2518f.json...
2026-03-06 00:50:53,333 - INFO - Processando Buster609_King743_53d82273-242b-438d-8056-adc9dc3f3078.json...
2026-03-06 00:50:53,502 - INFO - Processando Byron202_Vandervort697_5cb3f714-91f4-4b9f-bd58-dc64d8dd3a04.json...
2026-03-06 00:50:53,

2026-03-06 00:51:06,237 - INFO - Processando Chuck784_Nader710_669cd39f-c5fc-41c3-aff3-1ed7be7d40a7.json...
2026-03-06 00:51:06,409 - INFO - Processando Chung121_Koch169_8027e27d-405f-490c-a946-91d1b40c8fd0.json...
2026-03-06 00:51:06,592 - INFO - Processando Cicely661_Parker433_75d3758c-f568-448f-bb24-9699dc5732e1.json...
2026-03-06 00:51:06,773 - INFO - Processando Clair921_Weimann465_614b9e91-dcbd-4db4-9302-1d7fecac2bed.json...
2026-03-06 00:51:06,948 - INFO - Processando Clarence5_Abernathy524_3f31e949-d916-476d-bf37-f6874ab7b1c1.json...
2026-03-06 00:51:07,133 - INFO - Processando Claudia969_González124_6e4e4311-55f3-4c2d-8576-d4cb63f121b6.json...
2026-03-06 00:51:07,301 - INFO - Processando Claudine313_Schuppe920_a9df73f7-dd25-4b5b-b924-938edabfb461.json...
2026-03-06 00:51:07,491 - INFO - Processando Claudio955_Contreras711_f5b5e542-7999-45ed-bd61-3e21856ab21c.json...
2026-03-06 00:51:07,668 - INFO - Processando Clay913_Schaden604_cd9c52ca-a6d7-4eb4-b676-8208af287e75.json...
202

2026-03-06 00:51:19,910 - INFO - Processando Despina962_Collier206_c3127664-ea4c-482c-953a-92f24da84bba.json...
2026-03-06 00:51:20,096 - INFO - Processando Dessie725_Mohr916_55af98d6-fa0f-4c10-b249-8d10f59ef64d.json...
2026-03-06 00:51:20,270 - INFO - Processando Devora529_Hills818_55d2706c-bf82-4ecb-90c6-d7f7e121c4b6.json...
2026-03-06 00:51:20,447 - INFO - Processando Diego848_Colón139_83a9a02b-5013-4b66-be6f-6515eed1505f.json...
2026-03-06 00:51:20,620 - INFO - Processando Digna973_Powlowski563_808dc4fb-3abc-4336-a780-17863facddfa.json...
2026-03-06 00:51:20,803 - INFO - Processando Dinah304_Howe413_e9db3961-fe00-427f-a614-8820c1b55619.json...
2026-03-06 00:51:20,982 - INFO - Processando Dino214_Hoeger474_f56b73c1-4c29-4380-aa6b-623c5b1e0091.json...
2026-03-06 00:51:21,165 - INFO - Processando Dionne995_Jones311_1f780a7a-bdc5-406a-a6d9-6a349187110c.json...
2026-03-06 00:51:21,349 - INFO - Processando Domingo513_Sawayn19_a7f776dd-ad8a-4008-a0b8-f9b47b88570f.json...
2026-03-06 00:51:

...

2026-03-06 00:52:45,272 - INFO - Processando Magan944_Bergnaum523_e978a995-4b48-4c1b-8bd3-629b98b6b6b0.json...
2026-03-06 00:52:45,464 - INFO - Processando Magda24_Beatty507_1cb69366-0292-4e60-9f5f-2a9185a21020.json...
2026-03-06 00:52:45,644 - INFO - Processando Magda24_Harvey63_533bb688-e729-4394-a703-063f4bab21e9.json...
2026-03-06 00:52:45,835 - INFO - Processando Magdalena964_Koelpin146_f8bd603a-d34a-4536-8202-b571a2607f99.json...
2026-03-06 00:52:46,001 - INFO - Processando Man114_Herzog843_6d13dea8-401d-42d1-8451-1b3e78ead850.json...
2026-03-06 00:52:46,174 - INFO - Processando Manuel446_Luettgen772_a0a2069d-3abe-4591-a0de-7fffc37ad90d.json...
2026-03-06 00:52:46,349 - INFO - Processando Manuel446_Walker122_a28f848d-fb7e-4cf9-9b21-0d196954d809.json...
2026-03-06 00:52:46,530 - INFO - Processando Manuela585_Soria950_be72598d-f160-4c30-b285-59504f265315.json...
2026-03-06 00:52:46,731 - INFO - Processando Marc757_Ward668_a726931f-ebb7-485f-ba92-de7ec35a000b.json...
2026-03-06 00:5

2026-03-06 00:53:10,310 - INFO - Processando Michel472_Rath779_85c33295-33a5-44cc-864a-8eb29bd6f261.json...
2026-03-06 00:52:58,928 - INFO - Processando Mickey576_Gerhold939_b501777d-240e-4a5c-b611-d94f1965349d.json...
2026-03-06 00:52:59,110 - INFO - Processando Miguel_Ángel46_Vallejo39_8c631443-8b36-4660-ac2c-6d2862ea6d82.json...
2026-03-06 00:52:59,288 - INFO - Processando Mika226_Ryan260_3a39e3aa-c627-4787-8a8f-37759dfeb868.json...
2026-03-06 00:52:59,460 - INFO - Processando Mikki421_Baumbach677_ef772f4e-4bd2-4d76-a576-f63baea4e89c.json...
2026-03-06 00:53:11,192 - INFO - Processando Mikki421_King743_b539d641-64ad-4e5c-82c5-eea0bddacf45.json...
2026-03-06 00:53:11,375 - INFO - Processando Miles206_Gutkowski940_1b37ae9f-f895-4f71-91b8-22bad1ed333a.json...
2026-03-06 00:53:00,016 - INFO - Processando Miles206_Wyman904_c21fd57c-71cb-44ff-bb9f-fe85a23ba15a.json...
2026-03-06 00:53:00,200 - INFO - Processando Milford485_Hamill307_01cb4df5-2600-456d-9b29-767fd4f0f7d8.json...
2026-03-06 

2026-03-06 00:53:13,150 - INFO - Processando Otis335_Johns824_18f154cc-ec1c-427c-9ecc-48bae325c4d6.json...
2026-03-06 00:53:13,339 - INFO - Processando Ozzie259_Cummings51_5bff9560-f815-4e21-a72d-84e40413bace.json...
2026-03-06 00:53:13,519 - INFO - Processando Ozzie259_Walker122_cb9cd90f-77bf-4220-a97f-cde8e0e21878.json...
2026-03-06 00:53:13,736 - INFO - Processando Pablo44_Goyette777_10bf8a43-119e-43b7-9988-9e769f2200e6.json...
2026-03-06 00:53:13,922 - INFO - Processando Particia365_Gleason633_0060910b-52bd-44fc-8735-05012c94a696.json...
2026-03-06 00:53:14,118 - INFO - Processando Pat3_Considine820_4d1e6882-a78a-469f-8100-1dd60ba23d45.json...
2026-03-06 00:53:14,331 - INFO - Processando Patrick786_Moore224_5420cb2b-2a24-4542-8388-3659c5f3d82a.json...
2026-03-06 00:53:14,606 - INFO - Processando Patti584_Walsh511_7199010e-4f54-4027-875d-4277b4737fb4.json...
2026-03-06 00:53:26,353 - INFO - Processando Paula289_Kuhlman484_a3b90a9b-9cc4-493c-b646-2a4edd24d23e.json...
2026-03-06 00:53

2026-03-06 00:53:28,239 - INFO - Processando Ronni748_Wintheiser220_1b93c7d3-4427-488d-bba4-bee6d6e1c821.json...
2026-03-06 00:53:28,404 - INFO - Processando Ronnie7_Graham902_27e974ae-a79e-42cd-8ead-fdf31942ed20.json...
2026-03-06 00:53:28,571 - INFO - Processando Roosevelt595_Haley279_6fde96c6-95a8-4008-a98e-c7bece36e17c.json...
2026-03-06 00:53:40,296 - INFO - Processando Rory188_Bauch723_3c2086bb-32e7-4d82-ba8d-fd981a57123f.json...
2026-03-06 00:53:28,925 - INFO - Processando Rosanne862_Hoeger474_2bb235fc-a5a6-4a86-8b74-fd6b9d9c67d7.json...
2026-03-06 00:53:29,111 - INFO - Processando Rosario163_Canales95_a8296af6-7706-497b-be46-f5c101e7ce66.json...
2026-03-06 00:53:29,285 - INFO - Processando Rose199_Haley279_fe739bef-5e74-4f88-ae3c-9e7f5ed183c8.json...
2026-03-06 00:53:29,453 - INFO - Processando Roselia779_Lind531_99285aac-e5e3-4f5b-857d-f67271c97304.json...
2026-03-06 00:53:29,630 - INFO - Processando Ross213_Bayer639_d084de98-b1f8-4b37-a91a-e6c5cc3e5d20.json...
2026-03-06 00:5

['Aaron697_Brekke496_2fa15bc7-8866-461a-9000-f739e425860a.json',
 'Aaron697_Stiedemann542_41166989-975d-4d17-b9de-17f94cb3eec1.json',
 'Abby752_Kuvalis369_2b083021-e93f-4991-bf49-fd4f20060ef8.json',
 'Abel832_Connelly992_29e51479-f742-4474-8f8e-d2607d5269f6.json',
 'Abraham100_Heller342_262b819a-5193-404a-9787-b7f599358035.json',
 'Adam631_Cronin387_aff8f143-2375-416f-901d-b0e4c73e3e58.json',
 'Adam631_Shields502_9e2653fc-49e0-4b2e-86f8-e664bbe07be3.json',
 'Adelaida985_Gulgowski816_86662c9c-fcc8-41a6-a2e8-61270bf8a3b0.json',
 "Adolfo777_O'Keefe54_51d71f5e-96e5-4bc0-a656-e27e10dd4f32.json",
 'Adolph80_Turcotte120_52b1b75f-2b8a-4319-9542-7abc39502cab.json',
 'Adolph80_Williamson769_2de9315e-3784-4802-944a-926bcd8febd4.json',
 'Adria871_Ankunding277_89d97565-2497-4d7b-91b0-90543c0e6a9c.json',
 'Agnes294_Jenkins714_185d26ad-fb9f-40ae-afb0-94d72827d887.json',
 'Agustín529_González124_deb78d57-823a-45a0-81f2-68bdb5f9ee3a.json',
 'Ahmad985_Nader710_71e3d0af-39d4-416c-9bef-12c14f3bb821.json',